# Demo - Self-Intersection Resolving

This is a minimal setup to resolve self-intersection as in the paper (Sec. 4.2).

This notebook requires more dependencies than the `dvx-python` module. Take sure to install the dependencies:

````powershell
python -m venv .env
.env/Scripts/Activate.ps1 # depends on your os
pip install -r demos/requirements.txt
````

In [ ]:
import torch
import tqdm
import dvx.torch as dvx
import arap # as-rigid-as-possible energy in torch
import util # utility functions to keep the notebook clean
import polyscope as ps

torch.manual_seed(0);

## Setup parameters

In [ ]:
### configuration
mesh_path = "./meshes/siggraph_intersections.obj" # input mesh with self-intersections
d = 128                                            # voxel resolution
step = 0.01                                        # learning rate
steps = 10000                                      # maximum number of optimization steps
alpha = 0.005                                      # weight of arap loss

# initialize
V, F = util.load_obj(mesh_path)
V -= V.mean(0)
V /= torch.linalg.vector_norm(V, dim=1).max() * 1.25 # we leave some space for deformation
V_init = V.clone()
V.requires_grad_(True);

## Loss definition

In [ ]:
def loss_fn(eps=1e-3) -> torch.Tensor:
  # deformation loss
  loss_deform = arap.ARAP_energy(V, V_init, F)

  # overlap loss
  V_voxel = dvx.voxelize(d,V,F)
  intersections = V_voxel[V_voxel > 1.00 + eps]
  loss_intersect = (intersections).pow(2).sum()
  intersecting_voxels = intersections.size(0)

  loss = 1./(d**3) * loss_intersect + alpha * loss_deform
  return loss, intersecting_voxels

## Optimization Loop

In [ ]:
loss, intersecting_voxels = loss_fn()
print(f"Initial loss: {loss:.5f}. Intersecting voxels: {intersecting_voxels}.")

# optimize
itr = tqdm.trange(steps, desc=f"Loss: nan")
optim = torch.optim.Adam([V], lr=step)
losses = []
try:
  for t in itr:
    loss, intersecting_voxels = loss_fn()
    losses.append(loss.item())
    itr.set_description(f"Loss: {loss:.5f}. Intersecting voxels: {intersecting_voxels}.")
    if intersecting_voxels == 0: break
    optim.zero_grad()
    loss.backward()
    optim.step()
except KeyboardInterrupt:
  print(f"Skipping {steps - t} iterations.")

## Show results

In [ ]:
ps.init()
ps.register_surface_mesh("Initial surface", V_init.numpy(), F.numpy(), enabled=False)
ps.register_surface_mesh("Optimized surface", V.detach().numpy(), F.numpy())
ps.show()